# 07 — Comparison & Explainability

Run this **last**, after `00_Preprocessing.ipynb` and all 6 model
notebooks (`01`–`06`). Collects every model's saved results into one
comparison table + plots, then runs SHAP on the best ML model and best
DL model.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import tensorflow as tf

from utils import PROCESSED_DIR, RESULTS_DIR, MODELS_DIR, PLOTS_DIR, load_all_results

sns.set_theme(style="whitegrid")


## Load All Model Results

In [ ]:
all_results = load_all_results()
assert len(all_results) > 0, "No results found in results/ — run notebooks 01-06 first."
print(f"Loaded results for {len(all_results)} models:", [r['Model'] for r in all_results])


## Comparison Table

In [ ]:
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("Confusion Matrix", "history")}
    for r in all_results
])
results_df = results_df.sort_values("F1-Score", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(RESULTS_DIR, "comparison_table.csv"), index=False)

display(results_df.style.background_gradient(
    subset=["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"], cmap="Greens"
))


## Comparison Bar Charts (all 7 metrics)

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC",
                    "Train Time (s)", "Prediction Time (s)"]
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.ravel()
for ax, metric in zip(axes, metrics_to_plot):
    order = results_df.sort_values(metric, ascending=False)
    sns.barplot(data=order, x="Model", y=metric, hue="Model", palette="mako", legend=False, ax=ax)
    ax.set_title(f"{metric} Comparison")
    ax.tick_params(axis="x", rotation=30)
axes[-1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "comparison_bar_charts.png"), dpi=150)
plt.show()


## All Confusion Matrices Side-by-Side

In [ ]:
class_names = joblib.load(os.path.join(PROCESSED_DIR, "class_names.joblib"))

n_models = len(all_results)
n_cols = 3
n_rows = int(np.ceil(n_models / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, r in zip(axes, all_results):
    cm = np.array(r["Confusion Matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(r["Model"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.tick_params(axis="x", rotation=45)
for ax in axes[len(all_results):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "all_confusion_matrices.png"), dpi=150)
plt.show()


## Identify Best ML Model and Best DL Model

In [ ]:
ML_MODELS = {"Random Forest", "XGBoost", "LightGBM"}
DL_MODELS = {"MLP", "1D CNN", "Autoencoder+MLP"}

ml_results = [r for r in all_results if r["Model"] in ML_MODELS]
dl_results = [r for r in all_results if r["Model"] in DL_MODELS]

best_ml_name = max(ml_results, key=lambda r: r["F1-Score"])["Model"] if ml_results else None
best_dl_name = max(dl_results, key=lambda r: r["F1-Score"])["Model"] if dl_results else None
print(f"Best ML model: {best_ml_name}")
print(f"Best DL model: {best_dl_name}")


## Load Data + Best Models for SHAP

In [ ]:
X_train_res = np.load(os.path.join(PROCESSED_DIR, "X_train_res.npy"))
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
feature_names = joblib.load(os.path.join(PROCESSED_DIR, "feature_names.joblib"))
n_classes = joblib.load(os.path.join(PROCESSED_DIR, "n_classes.joblib"))

ML_MODEL_FILES = {"Random Forest": "RandomForest.joblib", "XGBoost": "XGBoost.joblib", "LightGBM": "LightGBM.joblib"}
DL_MODEL_FILES = {"MLP": "MLP.keras", "1D CNN": "CNN1D.keras", "Autoencoder+MLP": "AutoencoderMLP_classifier.keras"}

best_ml_model = joblib.load(os.path.join(MODELS_DIR, ML_MODEL_FILES[best_ml_name]))
best_dl_model = tf.keras.models.load_model(os.path.join(MODELS_DIR, DL_MODEL_FILES[best_dl_name]))

if best_dl_name == "Autoencoder+MLP":
    ae_encoder = tf.keras.models.load_model(os.path.join(MODELS_DIR, "AutoencoderMLP_encoder.keras"))

non_benign = [i for i, c in enumerate(class_names) if c.upper() != "BENIGN"]
SHAP_CLASS_IDX = non_benign[0] if non_benign else 0
print(f"Explaining class: {class_names[SHAP_CLASS_IDX]}")


## SHAP — Best ML Model

In [ ]:
X_test_sample = X_test[:200]

tree_explainer = shap.TreeExplainer(best_ml_model)
shap_values_ml = tree_explainer(X_test_sample)

if shap_values_ml.values.ndim == 3:
    class_shap_values = shap_values_ml.values[:, :, SHAP_CLASS_IDX]
else:
    class_shap_values = shap_values_ml.values

shap.summary_plot(class_shap_values, X_test_sample, feature_names=feature_names, show=False)
plt.title(f"SHAP Summary — {best_ml_name} (class: {class_names[SHAP_CLASS_IDX]})")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_summary_best_ml.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
if shap_values_ml.values.ndim == 3:
    single_explanation = shap_values_ml[0, :, SHAP_CLASS_IDX]
else:
    single_explanation = shap_values_ml[0]

shap.plots.waterfall(single_explanation, show=False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_waterfall_best_ml.png"), dpi=150, bbox_inches="tight")
plt.show()


## SHAP — Best DL Model

In [ ]:
background = X_train_res[np.random.choice(X_train_res.shape[0], 50, replace=False)]
X_test_sample_dl = X_test[:50]

def predict_fn(x):
    if best_dl_name == "1D CNN":
        x = x.reshape((x.shape[0], x.shape[1], 1))
    elif best_dl_name == "Autoencoder+MLP":
        x = ae_encoder.predict(x, verbose=0)
    return best_dl_model.predict(x, verbose=0)

kernel_explainer = shap.KernelExplainer(predict_fn, background)
shap_values_dl = kernel_explainer.shap_values(X_test_sample_dl, nsamples=100)

if isinstance(shap_values_dl, list):
    class_shap_values_dl = shap_values_dl[SHAP_CLASS_IDX]
else:
    class_shap_values_dl = shap_values_dl[:, :, SHAP_CLASS_IDX] if shap_values_dl.ndim == 3 else shap_values_dl

shap.summary_plot(class_shap_values_dl, X_test_sample_dl, feature_names=feature_names, show=False)
plt.title(f"SHAP Summary — {best_dl_name} (class: {class_names[SHAP_CLASS_IDX]})")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_summary_best_dl.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
expected_value = kernel_explainer.expected_value
expected_value = expected_value[SHAP_CLASS_IDX] if isinstance(expected_value, (list, np.ndarray)) else expected_value

waterfall_explanation = shap.Explanation(
    values=class_shap_values_dl[0], base_values=expected_value,
    data=X_test_sample_dl[0], feature_names=feature_names,
)
shap.plots.waterfall(waterfall_explanation, show=False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_waterfall_best_dl.png"), dpi=150, bbox_inches="tight")
plt.show()


## Final Summary

In [ ]:
print("=== FINAL COMPARISON ===")
display(results_df)
print(f"\nBest ML model: {best_ml_name}")
print(f"Best DL model: {best_dl_name}")
print(f"\nAll plots saved to: {PLOTS_DIR}/")
print(f"Comparison table saved to: {os.path.join(RESULTS_DIR, 'comparison_table.csv')}")
